# Actor Clustering - Louvain

This notebook implements the actor-level clustering pipeline for one model using the shared preprocessing and graph construction logic.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.clustering.louvain import run_louvain
from src.preprocessing.pipeline import prepare_common_actor_data
from src.utils.config import RESULTS_DIR, SEED
from src.utils.io_utils import save_json, save_results_csv

MAX_NODES = 2000
common = prepare_common_actor_data(max_nodes=MAX_NODES)
assignments, metrics = run_louvain(common["graph"], seed=SEED)
result_dir = RESULTS_DIR / "louvain"
save_results_csv(assignments, result_dir / "cluster_assignments.csv")
save_json(metrics, result_dir / "metrics.json")
print(metrics)

{'method': 'louvain', 'n_nodes': 2000, 'n_edges': 1031, 'n_clusters': 1362, 'modularity': 0.9247447784686685, 'runtime_seconds': 0.2840647870034445, 'seed': 42, 'resolution': 1.0}


In [2]:
from src.clustering.evaluation import evaluate_clusters
from src.clustering.visualization import save_cluster_figures, save_graph_partition_figure

labels = common["actor_df"].set_index("actor_id")["class"].reindex(assignments["actor_id"])
metrics.update(evaluate_clusters(assignments, labels=labels))
metrics["research_questions"] = [
    "Do frequently interacting addresses form stable communities consistent with an address-level entity proxy?",
    "How does AddrAddr community detection compare with learned graph embeddings?",
]
save_json(metrics, result_dir / "metrics.json")
save_cluster_figures(assignments, result_dir, title="Louvain actor communities")
save_graph_partition_figure(common["graph"], assignments, result_dir / "graph_partition.png", "Louvain partition on AddrAddr graph")
print("Research interpretation metrics:", {key: metrics[key] for key in ("modularity", "ari", "nmi", "ami")})

Research interpretation metrics: {'modularity': 0.9247447784686685, 'ari': 0.0006022152678985098, 'nmi': 0.017050733081720882, 'ami': 0.0042334792741784665}
